# Cruzamento Domínio x DFE/SAT SC

Este notebook pede os dois arquivos separadamente, compara as NF-es da DFE com as Entradas do Domínio e gera uma planilha com as notas faltantes e diferenças de valor.

**Envie primeiro o arquivo de Entradas do Domínio** (`.pdf`, `.xls` ou `.xlsx`) e depois a planilha **DFE/SAT**.

In [ ]:
!pip install -q pandas openpyxl xlrd pypdf reportlab

import html
import io
import re
import shutil
import subprocess
import unicodedata
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
from pypdf import PdfReader
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.platypus import Flowable, HRFlowable, PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle
from google.colab import files

pd.set_option('display.max_columns', 120)

In [ ]:
def sem_acento(texto):
    texto = '' if pd.isna(texto) else str(texto)
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(ch for ch in texto if not unicodedata.combining(ch))
    texto = re.sub(r'\s+', ' ', texto).strip().lower()
    return texto


def nome_coluna(col):
    return re.sub(r'[^a-z0-9]+', '_', sem_acento(col)).strip('_')


def tornar_colunas_unicas(cols):
    resultado = []
    vistos = {}
    for col in cols:
        base = nome_coluna(col) or 'coluna'
        vistos[base] = vistos.get(base, 0) + 1
        resultado.append(base if vistos[base] == 1 else f'{base}_{vistos[base]}')
    return resultado


def normalizar_numero_nf(valor):
    if pd.isna(valor):
        return ''
    texto = str(valor).strip()
    if texto.endswith('.0'):
        texto = texto[:-2]
    digitos = re.sub(r'\D+', '', texto)
    if not digitos:
        return ''
    return str(int(digitos))


def normalizar_serie(valor):
    if pd.isna(valor):
        return ''
    texto = str(valor).strip()
    if texto.endswith('.0'):
        texto = texto[:-2]
    digitos = re.sub(r'\D+', '', texto)
    if not digitos:
        return texto.upper()
    return str(int(digitos))


def normalizar_valor(valor):
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, str):
        valor = valor.strip().replace('.', '').replace(',', '.') if ',' in valor else valor.strip()
    return pd.to_numeric(valor, errors='coerce')


def normalizar_data_excel(serie):
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_datetime(serie, errors='coerce', unit='D', origin='1899-12-30')
    return pd.to_datetime(serie, errors='coerce', dayfirst=True)


def achar_coluna(df, candidatos, obrigatoria=True):
    colunas_norm = {col: sem_acento(col).replace('_', ' ') for col in df.columns}
    for candidato in candidatos:
        cand = sem_acento(candidato).replace('_', ' ')
        for col, norm in colunas_norm.items():
            if cand == norm or cand in norm:
                return col
    if obrigatoria:
        raise ValueError(f'Nao encontrei coluna esperada. Procurei por: {candidatos}. Colunas disponiveis: {list(df.columns)}')
    return None


def escolher_primeiro_upload(mensagem):
    print(mensagem)
    enviados = files.upload()
    if not enviados:
        raise ValueError('Nenhum arquivo enviado.')
    return next(iter(enviados.keys()))


def instalar_libreoffice_se_precisar():
    executavel = shutil.which('libreoffice') or shutil.which('soffice')
    if executavel:
        return executavel
    print('Instalando LibreOffice para converter o .xls antigo do Dominio...')
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'libreoffice'], check=True)
    executavel = shutil.which('libreoffice') or shutil.which('soffice')
    if not executavel:
        raise RuntimeError('Nao consegui instalar/localizar o LibreOffice para converter o .xls.')
    return executavel


def converter_xls_para_xlsx(caminho):
    caminho = Path(caminho)
    saida_dir = Path('arquivos_convertidos')
    saida_dir.mkdir(exist_ok=True)
    destino = saida_dir / f'{caminho.stem}.xlsx'
    if destino.exists():
        return str(destino)

    executavel = instalar_libreoffice_se_precisar()
    comando = [executavel, '--headless', '--convert-to', 'xlsx', '--outdir', str(saida_dir), str(caminho)]
    processo = subprocess.run(comando, text=True, capture_output=True)
    if processo.returncode != 0 or not destino.exists():
        detalhe = (processo.stderr or processo.stdout or '').strip()
        raise RuntimeError(f'Falha ao converter {caminho.name} para .xlsx. {detalhe}')
    print(f'Arquivo convertido para leitura: {destino}')
    return str(destino)


def preparar_excel_para_leitura(caminho):
    try:
        pd.ExcelFile(caminho)
        return caminho
    except Exception as erro:
        if Path(caminho).suffix.lower() == '.xls':
            print(f'O arquivo {Path(caminho).name} nao foi lido diretamente pelo pandas/xlrd.')
            print('Vou converter para .xlsx e continuar a conferencia.')
            return converter_xls_para_xlsx(caminho)
        raise erro


def encontrar_aba_e_cabecalho(caminho, termos):
    caminho_leitura = preparar_excel_para_leitura(caminho)
    xls = pd.ExcelFile(caminho_leitura)
    melhor = None
    for aba in xls.sheet_names:
        bruto = pd.read_excel(caminho_leitura, sheet_name=aba, header=None, dtype=object)
        for idx in range(min(30, len(bruto))):
            linha = ' | '.join(sem_acento(v) for v in bruto.iloc[idx].tolist())
            pontos = sum(1 for termo in termos if sem_acento(termo) in linha)
            if melhor is None or pontos > melhor['pontos']:
                melhor = {'aba': aba, 'header_idx': idx, 'pontos': pontos, 'bruto': bruto}
    if melhor is None or melhor['pontos'] == 0:
        raise ValueError(f'Nao consegui detectar cabecalho em {caminho}.')
    return melhor


def extrair_periodo_texto(texto):
    m = re.search(r'(\d{2}/\d{2}/\d{4})\s*(?:ate|até|a)\s*(\d{2}/\d{2}/\d{4})', texto, flags=re.IGNORECASE)
    if m:
        return f'{m.group(1)} ate {m.group(2)}'
    return ''


def extrair_periodo_nome_arquivo(caminho):
    nome = Path(caminho).name
    m = re.search(r'(\d{2})[-_/](\d{2})[-_/](\d{4})\s*a\s*(\d{2})[-_/](\d{2})[-_/](\d{4})', nome, flags=re.IGNORECASE)
    if not m:
        return ''
    inicio = f'{m.group(1)}/{m.group(2)}/{m.group(3)}'
    fim = f'{m.group(4)}/{m.group(5)}/{m.group(6)}'
    return f'{inicio} ate {fim}'


def extrair_empresa_pdf(texto):
    for linha in texto.splitlines():
        linha = ' '.join(str(linha).split())
        if not linha:
            continue
        if re.search(r'\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}|CNPJ|Per[ií]odo|P[aá]gina|Hora|Emiss[aã]o', linha, flags=re.IGNORECASE):
            continue
        return linha
    return ''


def extrair_metadados_entradas_bruto(bruto, header_idx):
    texto_cabecalho = ' '.join(str(v) for v in bruto.iloc[:header_idx].fillna('').to_numpy().ravel() if str(v).strip())
    empresa = ''
    for v in bruto.iloc[:header_idx].fillna('').to_numpy().ravel():
        v = ' '.join(str(v).split())
        if len(v) > 5 and not re.search(r'CNPJ|Insc|Per[ií]odo|ACOMPANHAMENTO', v, flags=re.IGNORECASE) and not re.search(r'\d{2}/\d{2}/\d{4}|\d{14}', v):
            empresa = v
            break
    return {'empresa': empresa, 'periodo_entradas': extrair_periodo_texto(texto_cabecalho)}


def carregar_entradas_pdf(caminho):
    texto = '\n'.join(page.extract_text() or '' for page in PdfReader(caminho).pages)
    metadados = {
        'empresa': extrair_empresa_pdf(texto),
        'periodo_entradas': extrair_periodo_texto(texto),
    }
    registros = []

    for numero_linha, linha in enumerate(texto.splitlines(), start=1):
        linha = ' '.join(str(linha).split())
        if not re.search(r'\d{2}/\d{2}/\d{4}', linha):
            continue

        m = re.search(
            r'^(?P<antes>.*?)(?P<data>\d{2}/\d{2}/\d{4})\s+'
            r'(?P<nota>\d+)\s+(?P<serie>\d+)\s+(?P<especie>\d+)\s+'
            r'(?P<resto>.*?)(?P<valor>[\d.]+,\d{2})$',
            linha,
        )
        if not m:
            continue

        fornecedor = ''
        uf = ''
        ac = ''
        cfop = ''
        f = re.search(r'(?:0,00\s*){3}(?P<fornecedor>.+?)\s*(?P<uf>[A-Z]{2})(?P<ac>\d{2})(?P<cfop>\d-\d{3})\s*$', m.group('resto'))
        if f:
            fornecedor = f.group('fornecedor').strip()
            uf = f.group('uf')
            ac = f.group('ac')
            cfop = f.group('cfop')

        registros.append({
            'linha_pdf': numero_linha,
            'data': m.group('data'),
            'nota': m.group('nota'),
            'serie': m.group('serie'),
            'especie': m.group('especie'),
            'fornecedor': fornecedor,
            'uf': uf,
            'ac': ac,
            'cfop': cfop,
            'valor_contabil': m.group('valor'),
            'linha_original_pdf': linha,
        })

    if not registros:
        raise ValueError('Nao encontrei linhas de entradas no PDF. Confira se o PDF e o relatorio Acompanhamento de Entradas do Dominio.')

    df = pd.DataFrame(registros)
    saida = pd.DataFrame()
    saida['dominio_numero'] = df['nota'].map(normalizar_numero_nf)
    saida['dominio_serie'] = df['serie'].map(normalizar_serie)
    saida['dominio_valor'] = df['valor_contabil'].map(normalizar_valor).round(2)
    saida['dominio_fornecedor'] = df['fornecedor']
    saida['dominio_data'] = pd.to_datetime(df['data'], errors='coerce', format='%d/%m/%Y')
    saida['dominio_aba_origem'] = 'PDF'
    saida['dominio_linha_pdf'] = df['linha_pdf']
    saida = saida[(saida['dominio_numero'] != '') & (saida['dominio_serie'] != '')].copy()
    saida.attrs.update(metadados)
    df.attrs.update(metadados)
    print(f'Linhas de entradas lidas do PDF: {len(saida)}')
    return saida, df


def carregar_entradas_dominio(caminho):
    if Path(caminho).suffix.lower() == '.pdf':
        return carregar_entradas_pdf(caminho)

    info = encontrar_aba_e_cabecalho(caminho, ['nota', 'serie', 'fornecedor', 'valor'])
    bruto = info['bruto']
    metadados = extrair_metadados_entradas_bruto(bruto, info['header_idx'])
    cabecalho = bruto.iloc[info['header_idx']].tolist()
    df = bruto.iloc[info['header_idx'] + 1:].copy()
    df.columns = tornar_colunas_unicas(cabecalho)
    df = df.dropna(how='all').reset_index(drop=True)
    manter = []
    for col in df.columns:
        coluna_generica = bool(re.match(r'^coluna(_\d+)?$', str(col)))
        manter.append((not coluna_generica) or df[col].notna().any())
    df = df.loc[:, manter]

    col_nota = achar_coluna(df, ['nota'])
    col_serie = achar_coluna(df, ['serie'])
    col_valor = achar_coluna(df, ['valor contabil', 'valor_contabil', 'valor'])
    col_fornecedor = achar_coluna(df, ['fornecedor'], obrigatoria=False)
    col_data = achar_coluna(df, ['data'], obrigatoria=False)

    saida = pd.DataFrame()
    saida['dominio_numero'] = df[col_nota].map(normalizar_numero_nf)
    saida['dominio_serie'] = df[col_serie].map(normalizar_serie)
    saida['dominio_valor'] = df[col_valor].map(normalizar_valor).round(2)
    if col_fornecedor:
        saida['dominio_fornecedor'] = df[col_fornecedor]
    if col_data:
        saida['dominio_data'] = normalizar_data_excel(df[col_data])
    saida['dominio_aba_origem'] = info['aba']
    saida = saida[(saida['dominio_numero'] != '') & (saida['dominio_serie'] != '')].copy()
    saida.attrs.update(metadados)
    df.attrs.update(metadados)
    return saida, df


def carregar_dfe(caminho):
    info = encontrar_aba_e_cabecalho(caminho, ['ChaveAcesso', 'NumeroDocumento', 'ValorTotalNota'])
    bruto = info['bruto']
    df = bruto.iloc[info['header_idx'] + 1:].copy()
    df.columns = [str(c).strip() for c in bruto.iloc[info['header_idx']].tolist()]
    df = df.dropna(how='all').reset_index(drop=True)

    col_chave = achar_coluna(df, ['ChaveAcesso', 'chave acesso'])
    col_numero = achar_coluna(df, ['NumeroDocumento', 'numero documento'])
    col_serie = achar_coluna(df, ['SerieDocumento', 'serie documento'])
    col_valor = achar_coluna(df, ['ValorTotalNota', 'valor total nota'])
    col_data = achar_coluna(df, ['DataEmissao', 'data emissao'], obrigatoria=False)
    col_situacao = achar_coluna(df, ['Situacao', 'situacao'], obrigatoria=False)
    col_tipo_operacao = achar_coluna(df, ['TipoDeOperacaoEntradaOuSaida', 'tipo operacao entrada saida'], obrigatoria=False)

    df['_dfe_chave'] = df[col_chave].astype(str).str.replace(r'\D+', '', regex=True)
    df['_dfe_numero'] = df[col_numero].map(normalizar_numero_nf)
    df['_dfe_serie'] = df[col_serie].map(normalizar_serie)
    df['_dfe_valor'] = df[col_valor].map(normalizar_valor).round(2)
    df['_dfe_situacao'] = df[col_situacao].astype(str).str.strip() if col_situacao else ''
    df['_dfe_tipo_operacao'] = df[col_tipo_operacao].astype(str).str.strip() if col_tipo_operacao else ''
    situacao_norm = df['_dfe_situacao'].map(sem_acento)
    tipo_norm = df['_dfe_tipo_operacao'].map(sem_acento)
    df['_dfe_cancelada'] = situacao_norm.str.contains('cancel', na=False)
    df['_dfe_marcada_entrada'] = tipo_norm.str.startswith('e', na=False) | tipo_norm.str.contains('entrada', na=False)
    df['_linha_dfe_origem'] = df.index + info['header_idx'] + 2

    chave_valida = df['_dfe_chave'].str.fullmatch(r'\d{44}', na=False)
    df_ignoradas = df[~chave_valida].copy()
    if not df_ignoradas.empty:
        df_ignoradas.insert(0, 'MotivoIgnorado', 'ChaveAcesso ausente ou invalida')
        print(f'Linhas da DFE ignoradas por nao terem chave valida: {len(df_ignoradas)}')

    df = df[chave_valida].copy().reset_index(drop=True)
    periodo_arquivo = extrair_periodo_nome_arquivo(caminho)
    if periodo_arquivo:
        df.attrs['periodo_dfe'] = periodo_arquivo
    if col_data and 'periodo_dfe' not in df.attrs:
        datas = pd.to_datetime(df[col_data], errors='coerce')
        if datas.notna().any():
            df.attrs['periodo_dfe'] = f'{datas.min().strftime("%d/%m/%Y")} ate {datas.max().strftime("%d/%m/%Y")}'
    if 'periodo_dfe' not in df.attrs:
        periodo_col = achar_coluna(df, ['PeriodoDeReferencia', 'periodo referencia'], obrigatoria=False)
        if periodo_col:
            periodos = sorted(str(v) for v in df[periodo_col].dropna().unique())
            if periodos:
                df.attrs['periodo_dfe'] = f'{periodos[0]} ate {periodos[-1]}'
    return df, df_ignoradas


def limpar_nome_comparacao(nome):
    texto = sem_acento(nome)
    texto = re.sub(r'[^a-z0-9 ]+', ' ', texto)
    palavras_ignoradas = {
        'ltda', 'me', 'eireli', 'sa', 's', 'a', 'epp', 'e', 'de', 'da', 'do', 'das', 'dos',
        'comercio', 'com', 'industria', 'ind', 'empresa', 'mat', 'materiais', 'produtos'
    }
    tokens = [t for t in texto.split() if t and t not in palavras_ignoradas]
    return ' '.join(tokens)


def similaridade_nomes(nome_a, nome_b):
    a = limpar_nome_comparacao(nome_a)
    b = limpar_nome_comparacao(nome_b)
    if not a or not b:
        return np.nan
    if a in b or b in a:
        return 1.0
    tokens_a = set(a.split())
    tokens_b = set(b.split())
    token_score = (2 * len(tokens_a & tokens_b) / (len(tokens_a) + len(tokens_b))) if tokens_a and tokens_b else 0
    ratio_score = SequenceMatcher(None, a, b).ratio()
    return round(max(token_score, ratio_score), 4)


def cruzar(dfe, entradas, tolerancia=0.01):
    col_nome_emitente = achar_coluna(dfe, ['NomeEmitente', 'nome emitente'], obrigatoria=False)
    grupos = {}
    for _, row in entradas.iterrows():
        chave = (row['dominio_numero'], row['dominio_serie'])
        grupos.setdefault(chave, []).append(row.to_dict())

    status = []
    valor_dominio = []
    fornecedor_dominio = []
    diferenca = []
    similaridade_fornecedor = []

    for _, row in dfe.iterrows():
        chave = (row['_dfe_numero'], row['_dfe_serie'])
        nome_emitente = row.get(col_nome_emitente, '') if col_nome_emitente else ''
        candidatos = grupos.get(chave, [])
        if not candidatos:
            status.append('FALTANDO NAS ENTRADAS')
            valor_dominio.append(np.nan)
            fornecedor_dominio.append('')
            diferenca.append(np.nan)
            similaridade_fornecedor.append(np.nan)
            continue

        def pontuar(candidato):
            sim = similaridade_nomes(nome_emitente, candidato.get('dominio_fornecedor', ''))
            sim_ordenacao = sim if pd.notna(sim) else 0.5
            val_dom = candidato.get('dominio_valor')
            dif_abs = abs((row['_dfe_valor'] or 0) - (val_dom or 0)) if pd.notna(val_dom) and pd.notna(row['_dfe_valor']) else 999999
            return (sim_ordenacao, -dif_abs)

        melhor = max(candidatos, key=pontuar)
        val_dom = melhor.get('dominio_valor')
        sim_nome = similaridade_nomes(nome_emitente, melhor.get('dominio_fornecedor', ''))
        dif = round((row['_dfe_valor'] or 0) - (val_dom or 0), 2) if pd.notna(val_dom) and pd.notna(row['_dfe_valor']) else np.nan
        fornecedor_confere = pd.isna(sim_nome) or sim_nome >= 0.45
        valor_confere = pd.notna(dif) and abs(dif) <= tolerancia
        if valor_confere and fornecedor_confere:
            status.append('OK')
        elif not valor_confere and not fornecedor_confere:
            status.append('DIFERENCA DE VALOR E FORNECEDOR')
        elif not fornecedor_confere:
            status.append('CONFERIR FORNECEDOR')
        else:
            status.append('DIFERENCA DE VALOR')
        valor_dominio.append(val_dom)
        fornecedor_dominio.append(melhor.get('dominio_fornecedor', ''))
        diferenca.append(dif)
        similaridade_fornecedor.append(sim_nome)

    resultado = dfe.copy()
    resultado.attrs.update(dfe.attrs)
    resultado.insert(0, 'StatusConferencia', status)
    resultado.insert(1, 'ValorDominio', valor_dominio)
    resultado.insert(2, 'DiferencaDfeMenosDominio', diferenca)
    resultado.insert(3, 'SimilaridadeFornecedor', similaridade_fornecedor)
    resultado.insert(4, 'FornecedorDominio', fornecedor_dominio)
    return resultado


class RelatorioPDF(SimpleDocTemplate):
    def afterFlowable(self, flowable):
        if hasattr(flowable, '_bookmarkName'):
            self.canv.bookmarkPage(flowable._bookmarkName)
            self.canv.addOutlineEntry(flowable.getPlainText(), flowable._bookmarkName, getattr(flowable, '_bookmarkLevel', 0), closed=False)


class CampoChavesPDF(Flowable):
    def __init__(self, texto, altura=4.2 * cm):
        super().__init__()
        self.texto = texto
        self.altura = altura

    def wrap(self, availWidth, availHeight):
        self.largura = availWidth
        return availWidth, self.altura

    def draw(self):
        self.canv.acroForm.textfieldRelative(
            name='ChavesFaltantes',
            tooltip='Campo interativo com todas as chaves faltantes',
            value=self.texto,
            width=self.largura,
            height=self.altura,
            fieldFlags='multiline',
            maxlen=20000,
            fontName='Courier',
            fontSize=7,
            fillColor=colors.HexColor('#F8FAFC'),
            borderColor=colors.HexColor('#CBD5E1'),
            textColor=colors.HexColor('#111827'),
            forceBorder=True,
        )


def estilos_pdf():
    base = getSampleStyleSheet()
    return {
        'empresa': ParagraphStyle('empresa', parent=base['Normal'], fontName='Helvetica-Bold', fontSize=14, leading=17, textColor=colors.HexColor('#FFFFFF'), spaceAfter=2),
        'titulo': ParagraphStyle('titulo', parent=base['Title'], fontName='Helvetica-Bold', fontSize=22, leading=26, textColor=colors.HexColor('#FFFFFF'), spaceAfter=2),
        'subtitulo': ParagraphStyle('subtitulo', parent=base['Normal'], fontName='Helvetica', fontSize=9, leading=13, textColor=colors.HexColor('#EAF5FF'), spaceAfter=6),
        'secao': ParagraphStyle('secao', parent=base['Heading2'], fontName='Helvetica-Bold', fontSize=13, leading=16, textColor=colors.HexColor('#176AC6'), spaceBefore=12, spaceAfter=6),
        'texto': ParagraphStyle('texto', parent=base['Normal'], fontName='Helvetica', fontSize=8, leading=11, textColor=colors.HexColor('#1A2D42')),
        'pequeno': ParagraphStyle('pequeno', parent=base['Normal'], fontName='Helvetica', fontSize=6.5, leading=8, textColor=colors.HexColor('#1A2D42')),
        'mono': ParagraphStyle('mono', parent=base['Normal'], fontName='Courier', fontSize=6.5, leading=8, textColor=colors.HexColor('#1A2D42')),
        'link': ParagraphStyle('link', parent=base['Normal'], fontName='Helvetica', fontSize=8.5, leading=12, textColor=colors.HexColor('#176AC6'), spaceAfter=3),
        'kpi_label': ParagraphStyle('kpi_label', parent=base['Normal'], fontName='Helvetica', fontSize=7.5, leading=9, textColor=colors.HexColor('#4A6580')),
        'kpi_value': ParagraphStyle('kpi_value', parent=base['Normal'], fontName='Helvetica-Bold', fontSize=15, leading=18, textColor=colors.HexColor('#1A2D42')),
    }


def secao_pdf(texto, bookmark, estilos, nivel=0):
    p = Paragraph(texto, estilos['secao'])
    p._bookmarkName = bookmark
    p._bookmarkLevel = nivel
    return p


def fmt_moeda(valor):
    if pd.isna(valor):
        return '-'
    texto = f'R$ {float(valor):,.2f}'
    return texto.replace(',', 'X').replace('.', ',').replace('X', '.')


def fmt_data(valor):
    if pd.isna(valor):
        return '-'
    data = pd.to_datetime(valor, errors='coerce')
    return '-' if pd.isna(data) else data.strftime('%d/%m/%Y')


def fmt_pct(valor):
    if pd.isna(valor):
        return '-'
    return f'{float(valor) * 100:.0f}%'


def valor_linha(row, coluna, padrao=''):
    if not coluna or coluna not in row.index or pd.isna(row[coluna]):
        return padrao
    return row[coluna]


def texto_pdf(valor, estilos, mono=False, limite=90):
    if pd.isna(valor):
        valor = ''
    texto = str(valor)
    if len(texto) > limite:
        texto = texto[:limite - 3] + '...'
    return Paragraph(html.escape(texto), estilos['mono' if mono else 'pequeno'])


def tabela_pdf(cabecalhos, linhas, larguras, estilos):
    dados = [[Paragraph(html.escape(str(c)), estilos['pequeno']) for c in cabecalhos]] + linhas
    tabela = Table(dados, colWidths=larguras, repeatRows=1, hAlign='LEFT')
    tabela.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#EBF4FF')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.HexColor('#176AC6')),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('GRID', (0, 0), (-1, -1), 0.25, colors.HexColor('#C8DEF5')),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#F7FBFF')]),
        ('LEFTPADDING', (0, 0), (-1, -1), 4),
        ('RIGHTPADDING', (0, 0), (-1, -1), 4),
        ('TOPPADDING', (0, 0), (-1, -1), 3),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 3),
    ]))
    return tabela


def quadro_resumo_pdf(linhas, larguras, estilos):
    dados = []
    for linha in linhas:
        dados.append([Paragraph(str(c), estilos['texto']) for c in linha])
    tabela = Table(dados, colWidths=larguras, hAlign='LEFT')
    tabela.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor('#FFFFFF')),
        ('BOX', (0, 0), (-1, -1), 0.8, colors.HexColor('#C8DEF5')),
        ('INNERGRID', (0, 0), (-1, -1), 0.35, colors.HexColor('#EEF5FD')),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('TOPPADDING', (0, 0), (-1, -1), 8),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
        ('LEFTPADDING', (0, 0), (-1, -1), 8),
        ('RIGHTPADDING', (0, 0), (-1, -1), 8),
    ]))
    return tabela


def status_cfg_pdf(status):
    mapa = {
        'OK': ('OK', '#155724'),
        'FALTANDO NAS ENTRADAS': ('Faltante', '#7F1D1D'),
        'DIFERENCA DE VALOR': ('Dif. Valor', '#C05621'),
        'CONFERIR FORNECEDOR': ('Fornecedor', '#176AC6'),
        'DIFERENCA DE VALOR E FORNECEDOR': ('Dif.+Forn.', '#4C1D95'),
    }
    return mapa.get(str(status), (str(status), '#1A2D42'))


def status_pdf(status, estilos):
    rotulo, cor = status_cfg_pdf(status)
    return Paragraph(f'<font color="{cor}"><b>{html.escape(rotulo)}</b></font>', estilos['pequeno'])


def dashboard_header_pdf(empresa, periodo_entradas, periodo_dfe, estilos):
    bloco = [
        [Paragraph(html.escape(empresa), estilos['empresa'])],
        [Paragraph('Relatorio Dominio x DFE/SAT', estilos['titulo'])],
        [Paragraph(f'<b>Entradas:</b> {html.escape(periodo_entradas)} &nbsp;&nbsp; | &nbsp;&nbsp; <b>DFE/SAT:</b> {html.escape(periodo_dfe)}', estilos['subtitulo'])],
        [Paragraph(f'Conferencia fiscal por numero, serie, valor e semelhanca de fornecedor/emitente. Gerado em {datetime.now().strftime("%d/%m/%Y %H:%M")}.', estilos['subtitulo'])],
    ]
    tabela = Table(bloco, colWidths=[25.4 * cm], hAlign='LEFT')
    tabela.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor('#176AC6')),
        ('BOX', (0, 0), (-1, -1), 0, colors.HexColor('#176AC6')),
        ('LEFTPADDING', (0, 0), (-1, -1), 14),
        ('RIGHTPADDING', (0, 0), (-1, -1), 14),
        ('TOPPADDING', (0, 0), (-1, 0), 14),
        ('BOTTOMPADDING', (0, -1), (-1, -1), 14),
    ]))
    return tabela


def kpi_cards_pdf(cards, estilos):
    dados = [[Paragraph(f'<font color="#4A6580">{html.escape(label)}</font><br/><font color="#1A2D42" size="15"><b>{html.escape(str(valor))}</b></font>', estilos['texto']) for label, valor in cards]]
    tabela = Table(dados, colWidths=[5.0 * cm] * len(cards), hAlign='LEFT')
    tabela.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor('#FFFFFF')),
        ('BOX', (0, 0), (-1, -1), 0.8, colors.HexColor('#C8DEF5')),
        ('INNERGRID', (0, 0), (-1, -1), 0.35, colors.HexColor('#EEF5FD')),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('TOPPADDING', (0, 0), (-1, -1), 9),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 9),
        ('LEFTPADDING', (0, 0), (-1, -1), 9),
        ('RIGHTPADDING', (0, 0), (-1, -1), 9),
    ]))
    return tabela


def barra_conformidade_pdf(ok, total, estilos):
    pct_ok = 0 if not total else max(0, min(1, ok / total))
    largura_total = 16 * cm
    largura_ok = max(0.15 * cm, largura_total * pct_ok) if pct_ok > 0 else 0.15 * cm
    largura_restante = max(0.15 * cm, largura_total - largura_ok)
    barra = Table([['', '']], colWidths=[largura_ok, largura_restante], rowHeights=[0.22 * cm], hAlign='LEFT')
    barra.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (0, 0), colors.HexColor('#176AC6')),
        ('BACKGROUND', (1, 0), (1, 0), colors.HexColor('#E8F3FE')),
        ('BOX', (0, 0), (-1, -1), 0.25, colors.HexColor('#C8DEF5')),
    ]))
    return [Paragraph(f'<b>Conformidade:</b> {pct_ok:.0%} OK', estilos['texto']), Spacer(1, 0.08 * cm), barra]


def rodape_pdf(canvas, doc):
    canvas.saveState()
    canvas.setFont('Helvetica', 7)
    canvas.setFillColor(colors.HexColor('#4A6580'))
    canvas.drawString(doc.leftMargin, 0.7 * cm, 'Relatorio Dominio x DFE/SAT')
    canvas.drawRightString(doc.pagesize[0] - doc.rightMargin, 0.7 * cm, f'Pagina {doc.page}')
    canvas.restoreState()


def salvar_relatorio(resultado, entradas, dfe_ignoradas=None, caminho_saida='relatorio_dominio_x_dfe.pdf'):
    pendencias = resultado[resultado['StatusConferencia'] != 'OK'].copy()
    faltantes = resultado[resultado['StatusConferencia'] == 'FALTANDO NAS ENTRADAS'].copy()
    diferencas = resultado[resultado['StatusConferencia'].isin(['DIFERENCA DE VALOR', 'DIFERENCA DE VALOR E FORNECEDOR', 'CONFERIR FORNECEDOR'])].copy()
    canceladas = resultado[resultado.get('_dfe_cancelada', False) == True].copy() if '_dfe_cancelada' in resultado.columns else pd.DataFrame()
    marcadas_entrada = resultado[resultado.get('_dfe_marcada_entrada', False) == True].copy() if '_dfe_marcada_entrada' in resultado.columns else pd.DataFrame()
    faltantes_baixar = faltantes[faltantes.get('_dfe_cancelada', False) != True].copy() if '_dfe_cancelada' in faltantes.columns else faltantes.copy()
    qtd_ignoradas = 0 if dfe_ignoradas is None else len(dfe_ignoradas)
    estilos = estilos_pdf()
    empresa = entradas.attrs.get('empresa') or 'Empresa nao identificada'
    periodo_entradas = entradas.attrs.get('periodo_entradas') or 'Periodo das Entradas nao identificado'
    periodo_dfe = resultado.attrs.get('periodo_dfe') or 'Periodo da DFE nao identificado'
    valor_faltante = faltantes['_dfe_valor'].sum() if '_dfe_valor' in faltantes.columns else 0
    valor_faltante_baixar = faltantes_baixar['_dfe_valor'].sum() if '_dfe_valor' in faltantes_baixar.columns else 0

    col_chave = achar_coluna(resultado, ['ChaveAcesso', 'chave acesso'], obrigatoria=False)
    col_data = achar_coluna(resultado, ['DataEmissao', 'data emissao'], obrigatoria=False)
    col_numero = achar_coluna(resultado, ['NumeroDocumento', 'numero documento'], obrigatoria=False)
    col_serie = achar_coluna(resultado, ['SerieDocumento', 'serie documento'], obrigatoria=False)
    col_valor = achar_coluna(resultado, ['ValorTotalNota', 'valor total nota'], obrigatoria=False)
    col_emitente = achar_coluna(resultado, ['NomeEmitente', 'nome emitente'], obrigatoria=False)
    col_cnpj = achar_coluna(resultado, ['CnpjOuCpfDoEmitente', 'cnpj cpf emitente', 'CnpjDoEmitente'], obrigatoria=False)
    col_situacao = achar_coluna(resultado, ['Situacao', 'situacao'], obrigatoria=False)
    col_tipo_operacao = achar_coluna(resultado, ['TipoDeOperacaoEntradaOuSaida', 'tipo operacao entrada saida'], obrigatoria=False)

    def linhas_dfe(df):
        linhas = []
        for _, row in df.iterrows():
            linhas.append([
                status_pdf(row.get('StatusConferencia', ''), estilos),
                texto_pdf(valor_linha(row, col_chave), estilos, mono=True, limite=60),
                texto_pdf(valor_linha(row, col_numero), estilos),
                texto_pdf(valor_linha(row, col_serie), estilos),
                texto_pdf(fmt_data(valor_linha(row, col_data, np.nan)), estilos),
                texto_pdf(fmt_moeda(row.get('_dfe_valor')), estilos),
                texto_pdf(fmt_moeda(row.get('ValorDominio')), estilos),
                texto_pdf(fmt_moeda(row.get('DiferencaDfeMenosDominio')), estilos),
                texto_pdf(fmt_pct(row.get('SimilaridadeFornecedor')), estilos),
                texto_pdf(valor_linha(row, col_situacao, row.get('_dfe_situacao', '')), estilos, limite=24),
                texto_pdf(valor_linha(row, col_tipo_operacao, row.get('_dfe_tipo_operacao', '')), estilos, limite=18),
                texto_pdf(valor_linha(row, col_emitente), estilos, limite=52),
                texto_pdf(row.get('FornecedorDominio', ''), estilos, limite=52),
            ])
        return linhas

    doc = RelatorioPDF(caminho_saida, pagesize=landscape(A4), rightMargin=1.1 * cm, leftMargin=1.1 * cm, topMargin=1.1 * cm, bottomMargin=1.2 * cm)
    story = []
    ok_count = int((resultado['StatusConferencia'] == 'OK').sum())
    story.append(dashboard_header_pdf(empresa, periodo_entradas, periodo_dfe, estilos))
    story.append(Spacer(1, 0.22 * cm))
    story.append(kpi_cards_pdf([
        ('Notas faltantes', len(faltantes)),
        ('Valor faltante', fmt_moeda(valor_faltante)),
        ('Canceladas SAT/DFE', len(canceladas)),
        ('Marcadas Entrada', len(marcadas_entrada)),
        ('Notas OK', ok_count),
    ], estilos))
    story.append(Spacer(1, 0.18 * cm))
    for bloco in barra_conformidade_pdf(ok_count, len(resultado), estilos):
        story.append(bloco)
    story.append(Spacer(1, 0.25 * cm))

    resumo = [
        ['Indicador', 'Quantidade'],
        ['Total DFE com chave valida', len(resultado)],
        ['Notas OK', ok_count],
        ['Faltando nas Entradas', len(faltantes)],
        ['Valor faltante total', fmt_moeda(valor_faltante)],
        ['Valor faltante para baixar (sem canceladas)', fmt_moeda(valor_faltante_baixar)],
        ['Notas canceladas SAT/DFE', len(canceladas)],
        ['Notas marcadas como Entrada na DFE/SAT', len(marcadas_entrada)],
        ['Diferencas ou fornecedor para conferir', len(diferencas)],
        ['Linhas DFE ignoradas sem chave valida', qtd_ignoradas],
        ['Total de pendencias', len(pendencias)],
    ]
    story.append(tabela_pdf(resumo[0], [[texto_pdf(a, estilos), texto_pdf(b, estilos)] for a, b in resumo[1:]], [8 * cm, 3 * cm], estilos))
    story.append(Spacer(1, 0.25 * cm))
    story.append(Paragraph('<b>Sumario interativo</b>', estilos['texto']))
    for label, target in [
        ('Criterios de conferencia', 'criterios'),
        ('Chaves faltantes para baixar', 'chaves'),
        ('Notas faltantes', 'faltantes'),
        ('Notas canceladas SAT/DFE', 'canceladas'),
        ('Notas marcadas como Entrada', 'marcadas_entrada'),
        ('Diferencas e fornecedores para conferir', 'diferencas'),
        ('Base DFE com status', 'base_dfe'),
        ('Entradas normalizadas', 'entradas'),
        ('Linhas ignoradas sem chave', 'ignoradas'),
    ]:
        story.append(Paragraph(f'<link href="#{target}">{html.escape(label)}</link>', estilos['link']))

    story.append(secao_pdf('Criterios de conferencia', 'criterios', estilos))
    story.append(Paragraph('O cruzamento usa numero do documento e serie. Quando ha possibilidade de documentos com mesmo numero/serie, o relatorio tambem compara a semelhanca entre o NomeEmitente da DFE e o Fornecedor das Entradas do Dominio. A nota so fica OK quando o valor confere e o fornecedor tem semelhanca aceitavel.', estilos['texto']))
    story.append(HRFlowable(width='100%', color=colors.HexColor('#CBD5E1'), thickness=0.6, spaceBefore=8, spaceAfter=8))

    story.append(secao_pdf('Chaves faltantes para baixar', 'chaves', estilos))
    chaves = [str(v) for v in faltantes_baixar['_dfe_chave'].dropna().tolist() if str(v).strip()]
    if chaves:
        story.append(Paragraph('Campo interativo: clique dentro da caixa, selecione o texto e copie as chaves faltantes para baixar. Notas canceladas ficam em secao propria e nao entram nesta lista.', estilos['texto']))
        story.append(CampoChavesPDF('\n'.join(chaves), altura=max(2.2 * cm, min(6.5 * cm, 0.32 * cm * len(chaves) + 0.8 * cm))))
    else:
        story.append(Paragraph('Nao ha chaves faltantes.', estilos['texto']))

    larguras_dfe = [2.4 * cm, 5.0 * cm, 1.25 * cm, 0.9 * cm, 1.5 * cm, 1.55 * cm, 1.55 * cm, 1.45 * cm, 1.0 * cm, 1.8 * cm, 1.1 * cm, 3.2 * cm, 3.2 * cm]
    cab_dfe = ['Status', 'Chave', 'Nota', 'Serie', 'Emissao', 'Valor DFE', 'Valor Dom.', 'Dif.', 'Sim.', 'Situacao', 'Tipo', 'Emitente DFE', 'Fornecedor Dominio']

    story.append(secao_pdf('Notas faltantes', 'faltantes', estilos))
    story.append(tabela_pdf(cab_dfe, linhas_dfe(faltantes), larguras_dfe, estilos) if not faltantes.empty else Paragraph('Nenhuma nota faltante.', estilos['texto']))

    story.append(secao_pdf('Notas canceladas SAT/DFE', 'canceladas', estilos))
    story.append(Paragraph('Estas notas ficam visiveis para conferencia, mas nao entram na caixa de chaves para baixar quando estiverem canceladas.', estilos['texto']))
    story.append(tabela_pdf(cab_dfe, linhas_dfe(canceladas), larguras_dfe, estilos) if not canceladas.empty else Paragraph('Nenhuma nota cancelada identificada na DFE/SAT.', estilos['texto']))

    story.append(secao_pdf('Notas marcadas como Entrada na DFE/SAT', 'marcadas_entrada', estilos))
    story.append(Paragraph('Notas em que o campo de tipo de operacao veio marcado como E ou Entrada.', estilos['texto']))
    story.append(tabela_pdf(cab_dfe, linhas_dfe(marcadas_entrada), larguras_dfe, estilos) if not marcadas_entrada.empty else Paragraph('Nenhuma nota marcada como Entrada identificada na DFE/SAT.', estilos['texto']))

    story.append(secao_pdf('Diferencas e fornecedores para conferir', 'diferencas', estilos))
    story.append(tabela_pdf(cab_dfe, linhas_dfe(diferencas), larguras_dfe, estilos) if not diferencas.empty else Paragraph('Nenhuma diferenca de valor ou fornecedor encontrada.', estilos['texto']))

    story.append(PageBreak())
    story.append(secao_pdf('Base DFE com status', 'base_dfe', estilos))
    story.append(tabela_pdf(cab_dfe, linhas_dfe(resultado), larguras_dfe, estilos))

    story.append(secao_pdf('Entradas normalizadas', 'entradas', estilos))
    linhas_entradas = []
    for _, row in entradas.iterrows():
        linhas_entradas.append([
            texto_pdf(row.get('dominio_numero'), estilos),
            texto_pdf(row.get('dominio_serie'), estilos),
            texto_pdf(fmt_data(row.get('dominio_data')), estilos),
            texto_pdf(fmt_moeda(row.get('dominio_valor')), estilos),
            texto_pdf(row.get('dominio_fornecedor', ''), estilos, limite=70),
            texto_pdf(row.get('dominio_aba_origem', ''), estilos),
        ])
    story.append(tabela_pdf(['Nota', 'Serie', 'Data', 'Valor Dominio', 'Fornecedor', 'Origem'], linhas_entradas, [2 * cm, 1.4 * cm, 2 * cm, 2.2 * cm, 9.5 * cm, 2.2 * cm], estilos))

    story.append(secao_pdf('Linhas ignoradas sem chave', 'ignoradas', estilos))
    if dfe_ignoradas is not None and not dfe_ignoradas.empty:
        linhas_ign = []
        for _, row in dfe_ignoradas.iterrows():
            linhas_ign.append([
                texto_pdf(row.get('MotivoIgnorado', ''), estilos, limite=40),
                texto_pdf(valor_linha(row, col_numero), estilos),
                texto_pdf(valor_linha(row, col_serie), estilos),
                texto_pdf(fmt_moeda(normalizar_valor(valor_linha(row, col_valor, np.nan))), estilos),
                texto_pdf(valor_linha(row, col_emitente), estilos, limite=70),
            ])
        story.append(tabela_pdf(['Motivo', 'Nota', 'Serie', 'Valor', 'Emitente'], linhas_ign, [5 * cm, 2 * cm, 1.4 * cm, 2 * cm, 8 * cm], estilos))
    else:
        story.append(Paragraph('Nenhuma linha ignorada.', estilos['texto']))

    doc.build(story, onFirstPage=rodape_pdf, onLaterPages=rodape_pdf)
    return caminho_saida

In [ ]:
arquivo_entradas = escolher_primeiro_upload('1) Envie agora o arquivo ENTRADAS do Dominio (.pdf, .xls ou .xlsx):')
arquivo_dfe = escolher_primeiro_upload('2) Envie agora a planilha DFE/SAT (.xlsx):')

entradas, entradas_original = carregar_entradas_dominio(arquivo_entradas)
dfe, dfe_ignoradas = carregar_dfe(arquivo_dfe)
resultado = cruzar(dfe, entradas)
saida = salvar_relatorio(resultado, entradas, dfe_ignoradas)

print('Conferencia concluida.')
print(resultado['StatusConferencia'].value_counts(dropna=False))
if not dfe_ignoradas.empty:
    print(f'Linhas ignoradas por falta de chave valida: {len(dfe_ignoradas)}')
files.download(saida)

## Conferência rápida

Depois de baixar o PDF gerado, use o sumário clicável para navegar. A seção `Chaves faltantes para baixar` traz um campo interativo com todas as chaves faltantes para seleção e cópia.